# EDA: Реєстр платників ПДВ України

**Опис:** первинний огляд та підготовка даних реєстру платників ПДВ (Open Data).

**Мета:** перевірити структуру, якість даних і підготувати базову візуалізацію для подальшого аналізу.

## Імпорти
Стандартні бібліотеки для аналізу та візуалізації.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))

from src.config import (
    DATA_FILE, CSV_SEP, CSV_ENCODING, CSV_ON_BAD_LINES,
    DATE_FORMAT, PLOT_STYLE, PANDAS_MAX_COLUMNS,
    ANALYSIS_START_YEAR, ECONOMIC_EVENT_YEAR,
    LEGAL_FORMS
)

## Конфігурація
Налаштування відображення та стилю графіків.

In [ ]:
pd.set_option("display.max_columns", PANDAS_MAX_COLUMNS)
plt.style.use(PLOT_STYLE)

## Завантаження даних
Читаємо CSV з обробкою кодування та помилкових рядків.

In [ ]:
df = pd.read_csv(
    DATA_FILE,
    sep=CSV_SEP,
    encoding=CSV_ENCODING,
    on_bad_lines=CSV_ON_BAD_LINES,
)

df["dat_term"] = df["dat_term"].replace("null", pd.NA)
df["dat_reestr"] = pd.to_datetime(df["dat_reestr"], format=DATE_FORMAT, errors="coerce")

## Базовий огляд
Швидко перевіряємо перші рядки, типи полів та пропуски.

In [ ]:
print("Data head:")
df.head()

In [ ]:
print("Data info:")
df.info()

print("\nData missing values:")
df.isnull().sum()

## Візуалізація
Приклад: кількість реєстрацій за роками.

In [ ]:
registrations_by_year = (
    df.dropna(subset=["dat_reestr"])
    .assign(year=lambda x: x["dat_reestr"].dt.year)
    .groupby("year")
    .size()
    .sort_index()
 )

plt.figure(figsize=(10, 4))
registrations_by_year.plot(kind="bar", color="steelblue")
plt.title("Кiлькiсть реєстрацiй за роками")
plt.xlabel("Рiк")
plt.ylabel("Кiлькiсть")
plt.tight_layout()
plt.show()

## Гіпотеза 1: Сезонність реєстрацій

**Припущення:** Реєстрація нових платників ПДВ має чітко виражену сезонність із піками на початку кожного кварталу (січень, квітень, липень, жовтень).

In [ ]:
df_forms = df.dropna(subset=["dat_reestr"]).copy()
df_forms["year"] = df_forms["dat_reestr"].dt.year

def extract_form(name):
    name_upper = str(name).upper()
    for form_abbr in LEGAL_FORMS.keys():
        if form_abbr in name_upper:
            return form_abbr
    return "Інше"

df_forms["form"] = df_forms["name"].apply(extract_form)

forms_by_year = df_forms.groupby(["year", "form"]).size().unstack(fill_value=0)

print("Розподіл за роками та формами:")
print(forms_by_year)

before_event = df_forms[df_forms["year"] < ECONOMIC_EVENT_YEAR].groupby("form").size()
after_event = df_forms[df_forms["year"] >= ECONOMIC_EVENT_YEAR].groupby("form").size()

print(f"\nДо {ECONOMIC_EVENT_YEAR} року:")
print(before_event)
print(f"Всього: {before_event.sum()}")

print(f"\nПісля {ECONOMIC_EVENT_YEAR} року:")
print(after_event)
print(f"Всього: {after_event.sum()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

forms_by_year.plot(ax=axes[0], marker="o")
axes[0].set_title("Динаміка популярності форм за роками")
axes[0].set_xlabel("Рік")
axes[0].set_ylabel("Кількість реєстрацій")
axes[0].legend(title="Форма", loc="best")
axes[0].grid(True, alpha=0.3)

periods = [f"До {ECONOMIC_EVENT_YEAR}", f"Після {ECONOMIC_EVENT_YEAR}"]
pp_values = [before_event.get("ПП", 0), after_event.get("ПП", 0)]
tov_values = [before_event.get("ТОВ", 0), after_event.get("ТОВ", 0)]

x = range(len(periods))
width = 0.35

axes[1].bar([i - width/2 for i in x], pp_values, width, label="ПП", color="steelblue")
axes[1].bar([i + width/2 for i in x], tov_values, width, label="ТОВ", color="coral")
axes[1].set_title(f"Популярність ПП vs ТОВ до/після {ECONOMIC_EVENT_YEAR}")
axes[1].set_ylabel("Кількість реєстрацій")
axes[1].set_xticks(x)
axes[1].set_xticklabels(periods)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print("\nАналіз гіпотези")
if after_event.get("ПП", 0) < before_event.get("ПП", 0):
    print(f"ПП дійсно знизилась: {before_event.get('ПП', 0)} → {after_event.get('ПП', 0)}")
else:
    print(f"ПП не знизилась або зросла: {before_event.get('ПП', 0)} → {after_event.get('ПП', 0)}")

if after_event.get("ТОВ", 0) > before_event.get("ТОВ", 0):
    print(f"ТОВ дійсно зросла: {before_event.get('ТОВ', 0)} → {after_event.get('ТОВ', 0)}")
else:
    print(f"ТОВ не зросла: {before_event.get('ТОВ', 0)} → {after_event.get('ТОВ', 0)}")

## Гіпотеза 2: Зміна популярності організаційно-правових форм

**Припущення:** Популярність форми "Приватне підприємство" (ПП) серед нових платників ПДВ катастрофічно знизилася на користь "Товариств з обмеженою відповідальністю" (ТОВ) після 2010 року.

In [ ]:
df_forms = df.dropna(subset=["dat_reestr"]).copy()
df_forms["year"] = df_forms["dat_reestr"].dt.year

def extract_form(name):
    name_upper = str(name).upper()
    if "ТОВ" in name_upper:
        return "ТОВ"
    elif "ПП" in name_upper or "ПРИВАТНЕ ПІДПРИЄМСТВО" in name_upper:
        return "ПП"
    elif '"ЗАТ"' in name_upper or "ЗАТ" in name_upper:
        return "ЗАТ"
    elif "АТ" in name_upper or "АКЦІОНЕРНЕ" in name_upper:
        return "АТ"
    elif "ООО" in name_upper:
        return "ООО"
    else:
        return "Інше"

df_forms["form"] = df_forms["name"].apply(extract_form)

forms_by_year = df_forms.groupby(["year", "form"]).size().unstack(fill_value=0)

print("Розподіл за роками та формами:")
print(forms_by_year)

before_2010 = df_forms[df_forms["year"] < 2010].groupby("form").size()
after_2010 = df_forms[df_forms["year"] >= 2010].groupby("form").size()

print("\nДо 2010 року:")
print(before_2010)
print(f"Всього: {before_2010.sum()}")

print("\nПісля 2010 року:")
print(after_2010)
print(f"Всього: {after_2010.sum()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

forms_by_year.plot(ax=axes[0], marker="o")
axes[0].set_title("Динаміка популярності форм за роками")
axes[0].set_xlabel("Рік")
axes[0].set_ylabel("Кількість реєстрацій")
axes[0].legend(title="Форма", loc="best")
axes[0].grid(True, alpha=0.3)

periods = ["До 2010", "Після 2010"]
pp_values = [before_2010.get("ПП", 0), after_2010.get("ПП", 0)]
tov_values = [before_2010.get("ТОВ", 0), after_2010.get("ТОВ", 0)]

x = range(len(periods))
width = 0.35

axes[1].bar([i - width/2 for i in x], pp_values, width, label="ПП", color="steelblue")
axes[1].bar([i + width/2 for i in x], tov_values, width, label="ТОВ", color="coral")
axes[1].set_title("Популярність ПП vs ТОВ до/після 2010")
axes[1].set_ylabel("Кількість реєстрацій")
axes[1].set_xticks(x)
axes[1].set_xticklabels(periods)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print("\nАналіз гіпотези")
if after_2010.get("ПП", 0) < before_2010.get("ПП", 0):
    print(f"ПП дійсно знизилась: {before_2010.get('ПП', 0)} → {after_2010.get('ПП', 0)}")
else:
    print(f"ПП не знизилась або зросла: {before_2010.get('ПП', 0)} → {after_2010.get('ПП', 0)}")

if after_2010.get("ТОВ", 0) > before_2010.get("ТОВ", 0):
    print(f"ТОВ дійсно зросла: {before_2010.get('ТОВ', 0)} → {after_2010.get('ТОВ', 0)}")
else:
    print(f"ТОВ не зросла: {before_2010.get('ТОВ', 0)} → {after_2010.get('ТОВ', 0)}")


## Гіпотеза 3: Вплив економічних подій на реєстрацію

**Припущення:** Кількість нових реєстрацій платників ПДВ суттєво зросла після 2014 року порівняно з періодом 2010–2013 років.

In [ ]:
df_economic = df.dropna(subset=["dat_reestr"]).copy()
df_economic["year"] = df_economic["dat_reestr"].dt.year

registrations_per_year = df_economic.groupby("year").size()

period_before = df_economic[(df_economic["year"] >= ANALYSIS_START_YEAR) & (df_economic["year"] < ECONOMIC_EVENT_YEAR)]
period_after = df_economic[df_economic["year"] >= ECONOMIC_EVENT_YEAR]

count_before = len(period_before)
count_after = len(period_after)

years_before = period_before["year"].nunique()
years_after = period_after["year"].nunique()

avg_before = count_before / years_before if years_before > 0 else 0
avg_after = count_after / years_after if years_after > 0 else 0

print("Порівняння періодів")
print(f"\n{ANALYSIS_START_YEAR}-{ECONOMIC_EVENT_YEAR-1} роки:")
print(f"  Всього реєстрацій: {count_before:,}")
print(f"  Середньо на рік: {avg_before:,.0f}")

print(f"\nВід {ECONOMIC_EVENT_YEAR} року:")
print(f"  Всього реєстрацій: {count_after:,}")
print(f"  Середньо на рік: {avg_after:,.0f}")

change_percent = ((avg_after - avg_before) / avg_before * 100) if avg_before > 0 else 0
print(f"\nЗміна середньої кількості: {change_percent:+.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

years_list = registrations_per_year.index.tolist()
colors_bars = ['steelblue' if ANALYSIS_START_YEAR <= year < ECONOMIC_EVENT_YEAR else 'coral' if year >= ECONOMIC_EVENT_YEAR else 'gray' 
               for year in years_list]

registrations_per_year.plot(kind="bar", ax=axes[0], color=colors_bars)

if ECONOMIC_EVENT_YEAR in years_list:
    idx_event = years_list.index(ECONOMIC_EVENT_YEAR)
    axes[0].axvline(x=idx_event, color="black", linestyle="--", linewidth=2, label=f"{ECONOMIC_EVENT_YEAR} рік")

axes[0].set_title("Динаміка реєстрацій з виділенням періодів")
axes[0].set_xlabel("Рік")
axes[0].set_ylabel("Кількість реєстрацій")
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis="y")

periods = [f"{ANALYSIS_START_YEAR}-{ECONOMIC_EVENT_YEAR-1}", f"Від {ECONOMIC_EVENT_YEAR}"]
averages = [avg_before, avg_after]
colors = ["steelblue", "coral"]

axes[1].bar(periods, averages, color=colors, edgecolor="black")
axes[1].set_title("Середня кількість реєстрацій на рік")
axes[1].set_ylabel("Кількість реєстрацій")
axes[1].grid(True, alpha=0.3, axis="y")

for i, v in enumerate(averages):
    axes[1].text(i, v + v*0.02, f"{v:,.0f}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

print("\nАналіз гіпотези")
if avg_after > avg_before:
    print(f"Гіпотеза підтверджується: кількість реєстрацій зросла на {change_percent:.1f}%")
else:
    print(f"Гіпотеза не підтверджується: кількість реєстрацій змінилась на {change_percent:.1f}%")